# Bi-Scale (extended 21.5 h) — generate Phase-1 submission (Google Colab)

Generation only, no training.

**Before running**
1. Runtime → Change runtime type → **T4 GPU**.
2. Put these files in one Google Drive folder (default: `MyDrive/biscale21h/`):
   * the extended run's checkpoint: `stage2_done.pt` and/or `best_model.pt` (from the Kaggle run's Output → `checkpoints/`)
   * your tokenizer: `vocab.json`, `merges.txt` (Output → `tokenizer/`)
3. Colab secrets (key icon on the left): add `HF_TOKEN` = your *new* Hugging Face token, enable notebook access.
4. Runtime → **Run all**. Keep the tab open (free Colab disconnects idle sessions).

Result: `submission.zip` in the same Drive folder (and in `/content/work`). Expected time on one T4: ~1–1.5 h.

In [ ]:
# 1 — setup (Colab)
import os, json, glob, shutil, time, torch
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/biscale21h"
WORK = "/content/work"; os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
assert os.path.isdir(DRIVE_DIR), f"{DRIVE_DIR} not found — create it and upload the files"
print("files in Drive folder:", sorted(os.listdir(DRIVE_DIR)))

NGPU = torch.cuda.device_count()
assert NGPU >= 1, "no GPU — Runtime > Change runtime type > T4 GPU"
print("GPUs:", NGPU, [torch.cuda.get_device_name(i) for i in range(NGPU)])
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!pip install -q tokenizers huggingface_hub
from huggingface_hub import hf_hub_download
TEST_PATH = hf_hub_download("Exploration-Lab/CS-781-Capstone", "Phase_1_Pretraining/Test/test.json", repo_type="dataset")
print("test prompts:", len(json.load(open(TEST_PATH, encoding="utf-8"))))

In [ ]:
# 2 — tokenizer + checkpoint
TOK_DIR = f"{WORK}/tokenizer"; os.makedirs(TOK_DIR, exist_ok=True)
for fn in ["vocab.json", "merges.txt"]:
    hits = sorted(glob.glob(f"{DRIVE_DIR}/**/{fn}", recursive=True))
    assert hits, f"{fn} not found in the Drive folder — upload vocab.json and merges.txt"
    shutil.copy(hits[0], f"{TOK_DIR}/{fn}"); print("tokenizer:", hits[0])

cands = sorted(glob.glob(f"{DRIVE_DIR}/**/stage2_done.pt", recursive=True) +
               glob.glob(f"{DRIVE_DIR}/**/best_model.pt", recursive=True))
assert cands, "no stage2_done.pt / best_model.pt in the Drive folder — upload the extended run checkpoint"
info = []
for p in cands:
    ck = torch.load(p, map_location="cpu", weights_only=False)
    info.append((p, ck.get("val_loss"), ck.get("tokens")))
    print(f"{p}\n    val_loss={ck.get('val_loss')}  tokens={ck.get('tokens')}")
    del ck


CKPT_PATH = min((x for x in info if x[1] is not None), key=lambda x: x[1])[0]
print("\nusing:", CKPT_PATH)

In [ ]:
%%writefile model.py
"""model.py for the Bi-Scale generation notebook."""
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================== copied from the Bi-Scale training notebook ==============================
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq=2048, theta=10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self._build_cache(max_seq)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, device=self.inv_freq.device).float()
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

    def forward(self, seq_len):
        if seq_len > self.cos_cached.shape[2]:
            self._build_cache(seq_len)
        return self.cos_cached[:, :, :seq_len, :], self.sin_cached[:, :, :seq_len, :]

def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rotary(x, cos, sin):
    return x * cos + rotate_half(x) * sin

class BiScaleGQA(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, dropout, ctx_len, rope_theta, is_local=False, window_size=256):
        super().__init__()
        self.n_heads    = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_groups   = n_heads // n_kv_heads
        self.d_head     = d_model // n_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.o_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.d_head, max_seq=ctx_len, theta=rope_theta)

        causal = torch.tril(torch.ones(ctx_len, ctx_len))
        if is_local:
            causal = causal * torch.triu(torch.ones(ctx_len, ctx_len), diagonal=-window_size)
        self.register_buffer("mask", causal.view(1, 1, ctx_len, ctx_len))


        for proj in [self.q_proj, self.k_proj, self.v_proj, self.o_proj]:
            nn.init.normal_(proj.weight, mean=0.0, std=0.02)

    def forward(self, x):
        B, T, C = x.shape
        dh = self.d_head

        q = self.q_proj(x).view(B, T, self.n_heads, dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)

        cos, sin = self.rope(T)
        q = apply_rotary(q, cos, sin)
        k = apply_rotary(k, cos, sin)

        k = k.repeat_interleave(self.n_groups, dim=1)
        v = v.repeat_interleave(self.n_groups, dim=1)

        scale = dh ** -0.5
        att = (q @ k.transpose(-2, -1)) * scale
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = self.attn_drop(F.softmax(att, dim=-1))

        out = (att @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.o_proj(out)

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up   = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
        for proj in [self.gate, self.up, self.down]:
            nn.init.normal_(proj.weight, mean=0.0, std=0.02)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class LearnedGatedResidual(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Linear(dim, dim, bias=True)

        nn.init.normal_(self.gate.weight, mean=0.0, std=0.01)
        nn.init.constant_(self.gate.bias, 2.0)

    def forward(self, x, sublayer):
        g = torch.sigmoid(self.gate(x))
        return x + g * sublayer

class BiScaleLegalBlock(nn.Module):
    def __init__(self, cfg, layer_idx):
        super().__init__()
        is_local = (layer_idx < (cfg["n_layers"] // 2))
        self.norm1 = RMSNorm(cfg["d_model"])
        self.attn  = BiScaleGQA(
            cfg["d_model"], cfg["n_heads"], cfg["n_kv_heads"],
            cfg["dropout"], cfg["ctx_len"], cfg["rope_theta"],
            is_local=is_local, window_size=cfg["local_window"]
        )
        self.res1 = LearnedGatedResidual(cfg["d_model"])
        self.norm2 = RMSNorm(cfg["d_model"])
        self.ffn   = SwiGLU(cfg["d_model"], cfg["d_ff"])
        self.res2  = LearnedGatedResidual(cfg["d_model"])
        self.drop  = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        x = self.res1(x, self.drop(self.attn(self.norm1(x))))
        x = self.res2(x, self.drop(self.ffn(self.norm2(x))))
        return x

class BiScaleLegalLLaMA(nn.Module):
    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.ctx     = cfg["ctx_len"]
        self.tok_emb = nn.Embedding(vocab_size, cfg["d_model"])
        self.drop    = nn.Dropout(cfg["dropout"])
        self.blocks  = nn.ModuleList([BiScaleLegalBlock(cfg, i) for i in range(cfg["n_layers"])])
        self.norm    = RMSNorm(cfg["d_model"])
        self.lm_head = nn.Linear(cfg["d_model"], vocab_size, bias=False)


        self.lm_head.weight = self.tok_emb.weight


        nn.init.normal_(self.tok_emb.weight, mean=0.0, std=0.02)

    def forward(self, ids, targets=None):
        B, T = ids.shape
        x = self.drop(self.tok_emb(ids))
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss
# ======================================= end of copied code =======================================


_ORIGINAL_ATTN_FORWARD = BiScaleGQA.forward


def _sdpa_attn_forward(self, x):
    B, T, C = x.shape
    dh = self.d_head
    q = self.q_proj(x).view(B, T, self.n_heads, dh).transpose(1, 2)
    k = self.k_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
    v = self.v_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
    cos, sin = self.rope(T)
    q = apply_rotary(q, cos.to(q.dtype), sin.to(q.dtype))
    k = apply_rotary(k, cos.to(k.dtype), sin.to(k.dtype))
    k = k.repeat_interleave(self.n_groups, dim=1)
    v = v.repeat_interleave(self.n_groups, dim=1)
    y = F.scaled_dot_product_attention(q, k, v, attn_mask=self.mask[:, :, :T, :T].bool(),
                                       dropout_p=0.0, scale=dh ** -0.5)
    return self.o_proj(y.transpose(1, 2).contiguous().view(B, T, -1))


def use_sdpa(flag=True):
    BiScaleGQA.forward = _sdpa_attn_forward if flag else _ORIGINAL_ATTN_FORWARD


class BiScaleForHarness(BiScaleLegalLLaMA):
    def forward(self, input_ids, prefix_lens=None):
        return super().forward(input_ids)[0]


def build_model(model_cfg):
    use_sdpa(model_cfg.get("use_sdpa", True))
    return BiScaleForHarness(model_cfg, model_cfg["vocab_size"])

In [ ]:
%%writefile generate.py
import glob, json, os, re, sys, time, zipfile

import torch
import torch.distributed as dist
import torch.nn.functional as F

CFG = json.load(open(sys.argv[1]))
IN_PATH, OUT_PREFIX = sys.argv[2], sys.argv[3]
RUN, MODEL_CFG, GEN = CFG["run"], CFG["model"], CFG["gen"]
TARGET_WORDS = 128

if RUN.get("smoke_test"):
    import dummy_model as model_lib
else:
    import model as model_lib

if int(os.environ.get("WORLD_SIZE", "1")) > 1:
    dist.init_process_group("nccl")
    RANK, WORLD, LOCAL = dist.get_rank(), dist.get_world_size(), int(os.environ["LOCAL_RANK"])
else:
    RANK, WORLD, LOCAL = 0, 1, 0
torch.cuda.set_device(LOCAL)
DEV = torch.device("cuda", LOCAL)
torch.manual_seed(int(GEN.get("seed", 0)) + RANK)


def log(*a):
    if RANK == 0:
        print(*a, flush=True)


# ----------------------------------------------------------------------------- tokenizer + model
from tokenizers import ByteLevelBPETokenizer

tok = ByteLevelBPETokenizer(vocab=os.path.join(RUN["tokenizer_dir"], "vocab.json"),
                            merges=os.path.join(RUN["tokenizer_dir"], "merges.txt"))
CTX = int(CFG["train"]["ctx_len"])
MODEL_CFG.setdefault("vocab_size", tok.get_vocab_size())
MODEL_CFG.setdefault("max_seq_len", CTX)

ck_choice = GEN["checkpoint"]
ck_path = ck_choice if os.path.exists(ck_choice) else os.path.join(RUN["out_dir"], f"{RUN['run_name']}_{ck_choice}.pt")
model = model_lib.build_model(MODEL_CFG).to(DEV)
ck = torch.load(ck_path, map_location=DEV, weights_only=False)
model.load_state_dict(ck["model"])
model.eval()
log(f"Loaded {ck_path} (step {ck.get('step')}, val {ck.get('val_loss')})")
del ck
HAS_CACHE = hasattr(model, "prefill") and hasattr(model, "decode_step")
log("Decoding with KV cache (prefill/decode_step)" if HAS_CACHE else
    "No prefill/decode_step on the model: recomputing the full sequence each step (slower)")

# ----------------------------------------------------------------------------- sampling utils
MAX_NEW = int(GEN["max_new_tokens"])
PROMPT_MAX = CTX - MAX_NEW
assert PROMPT_MAX > 64, "ctx_len too small for max_new_tokens"
BAN = [tok.token_to_id(t) for t in ("<s>", "</s>", "<unk>", "<pad>", "<mask>") if tok.token_to_id(t) is not None]


class NgramBlocker:

    def __init__(self, n, prompt_ids, scope):
        self.n, self.seen = n, {}
        self.hist = list(prompt_ids[-(n - 1):]) if (n and scope != "all") else []
        if n and scope == "all":
            for i in range(len(prompt_ids) - n + 1):
                self.seen.setdefault(tuple(prompt_ids[i:i + n - 1]), set()).add(prompt_ids[i + n - 1])
            self.hist = list(prompt_ids[-(n - 1):])
        self.scope = scope
        self.gen_count = 0

    def banned(self):
        if not self.n or len(self.hist) < self.n - 1:
            return ()
        return self.seen.get(tuple(self.hist[-(self.n - 1):]), ())

    def push(self, t):
        if self.n:
            key = tuple(self.hist[-(self.n - 1):])

            if len(key) == self.n - 1 and (self.scope == "all" or self.gen_count >= self.n - 1):
                self.seen.setdefault(key, set()).add(t)
            self.hist.append(t)
            self.gen_count += 1


def pick_next(logits, gen_tokens, blockers):
    if BAN:
        logits[:, BAN] = -float("inf")
    rp = GEN["rep_penalty"]
    if rp and rp != 1.0 and gen_tokens.numel():
        sc = torch.gather(logits, 1, gen_tokens)
        logits.scatter_(1, gen_tokens, torch.where(sc < 0, sc * rp, sc / rp))
    for b, bl in enumerate(blockers):
        bad = bl.banned()
        if bad:
            logits[b, list(bad)] = -float("inf")
    T = GEN["temperature"]
    if not T:
        return logits.argmax(-1)
    logits = logits / T
    k = GEN.get("top_k") or 0
    if k:
        kth = torch.topk(logits, k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    p = GEN.get("top_p") or 1.0
    probs = F.softmax(logits, -1)
    if p < 1.0:
        sp, si = probs.sort(-1, descending=True)
        sp[(sp.cumsum(-1) - sp) > p] = 0.0
        probs = torch.zeros_like(probs).scatter_(1, si, sp)
    probs = probs / probs.sum(-1, keepdim=True)
    return torch.multinomial(probs, 1).squeeze(1)


def clean(text):
    return re.sub(r"\s+", " ", text).strip()


@torch.inference_mode()
def generate_batch(prompts_ids):
    B, L = len(prompts_ids), len(prompts_ids[0])
    ids = torch.tensor(prompts_ids, dtype=torch.long, device=DEV)
    plen = torch.full((B,), L, dtype=torch.long, device=DEV)
    blockers = [NgramBlocker(GEN.get("no_repeat_ngram") or 0, p, GEN.get("ngram_scope", "generated"))
                for p in prompts_ids]
    gen = torch.empty((B, 0), dtype=torch.long, device=DEV)
    done = torch.zeros(B, dtype=torch.bool, device=DEV)
    with torch.autocast("cuda", dtype=torch.float16):
        if HAS_CACHE:
            logits, cache = model.prefill(ids, plen)
        else:
            logits = model(ids, plen)[:, -1]
    for step in range(MAX_NEW):
        nxt = pick_next(logits.float(), gen, blockers)
        gen = torch.cat([gen, nxt[:, None]], 1)
        nl = nxt.tolist()
        for b, bl in enumerate(blockers):
            bl.push(nl[b])
        if (step + 1) % 16 == 0:
            for b in range(B):
                if not done[b] and len(tok.decode(gen[b].tolist()).split()) > TARGET_WORDS:
                    done[b] = True
            if bool(done.all()):
                break
        if step == MAX_NEW - 1:
            break
        with torch.autocast("cuda", dtype=torch.float16):
            if HAS_CACHE:
                logits, cache = model.decode_step(nxt[:, None], cache)
            else:
                logits = model(torch.cat([ids, gen], 1), plen)[:, -1]
    return [clean(tok.decode(g.tolist())) for g in gen]


# ----------------------------------------------------------------------------- run
data = json.load(open(IN_PATH, encoding="utf-8"))
all_ids = list(data.keys())
if RUN.get("smoke_test"):
    all_ids = all_ids[:16]
mine = all_ids[RANK::WORLD]
enc = {k: tok.encode(data[k]["text"]).ids[-PROMPT_MAX:] for k in mine}
order = sorted(mine, key=lambda k: len(enc[k]))
BS = int(GEN["batch_size"])
out, dropped = {}, 0
t0 = time.time()
for i in range(0, len(order), BS):
    keys = order[i:i + BS]
    m = min(len(enc[k]) for k in keys)
    dropped += sum(len(enc[k]) - m for k in keys)
    texts = generate_batch([enc[k][-m:] for k in keys])
    for k, t in zip(keys, texts):
        out[k] = t
    if RANK == 0 and (i // BS) % 5 == 0:
        el = time.time() - t0
        print(f"  rank0 {i+len(keys)}/{len(order)} prompts | {el/60:.1f} min | "
              f"eta {el/(i+len(keys))*(len(order)-i-len(keys))/60:.1f} min", flush=True)

part = f"{OUT_PREFIX}.part{RANK}.json"
json.dump(out, open(part, "w", encoding="utf-8"), ensure_ascii=False)
log(f"Context tokens trimmed for batching (rank0): {dropped} total")
if WORLD > 1:
    dist.barrier()

if RANK == 0:
    merged = {}
    for pf in sorted(glob.glob(f"{OUT_PREFIX}.part*.json")):
        merged.update(json.load(open(pf, encoding="utf-8")))
        os.remove(pf)
    preds, short = {}, 0
    for k in all_ids:
        w = merged.get(k, "").split()
        if len(w) < TARGET_WORDS:
            short += 1
        preds[k] = {"text": " ".join(w[:TARGET_WORDS])}
    pred_path = f"{OUT_PREFIX}_prediction.json"
    json.dump(preds, open(pred_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    zip_path = f"{OUT_PREFIX}_submission.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_path, arcname="prediction.json")
    print(f"Wrote {pred_path} and {zip_path} | {len(preds)} ids | {short} under {TARGET_WORDS} words | "
          f"{(time.time()-t0)/60:.1f} min", flush=True)
if WORLD > 1:
    dist.destroy_process_group()

In [ ]:
# 5 — config (architecture = the Bi-Scale training notebook's CFG; decoding = same as the Prefix-LM run)
BISCALE = dict(n_layers=16, n_heads=12, n_kv_heads=4, d_model=768, d_ff=2048, ctx_len=1024,
               local_window=256, dropout=0.1, rope_theta=10000.0)
GEN = dict(checkpoint=CKPT_PATH, max_new_tokens=256, batch_size=32,
           temperature=0, top_k=0, top_p=1.0, rep_penalty=1.0, no_repeat_ngram=4, ngram_scope="generated", seed=0)
CFG = dict(run=dict(run_name="biscale21h", tokenizer_dir=TOK_DIR, token_dir="", out_dir="/content/work/run",
                    smoke_test=False),
           model=BISCALE, train=dict(ctx_len=1024), gen=GEN)
os.makedirs("/content/work/run", exist_ok=True)
json.dump(CFG, open("gen_config.json", "w"), indent=1)
print(json.dumps(GEN, indent=1))

In [ ]:
# 6 — check: checkpoint loads exactly, and the fast attention gives the same logits as the original
import importlib, sys; sys.path.insert(0, "."); import model as M; importlib.reload(M)
from tokenizers import ByteLevelBPETokenizer
V = ByteLevelBPETokenizer(vocab=f"{TOK_DIR}/vocab.json", merges=f"{TOK_DIR}/merges.txt").get_vocab_size()
m = M.build_model(dict(BISCALE, vocab_size=V)).cuda().eval()
sd = torch.load(CKPT_PATH, map_location="cuda", weights_only=False)["model"]
m.load_state_dict(sd)
print(f"loaded, params {sum(p.numel() for p in m.parameters())/1e6:.2f}M")
x = torch.randint(5, V, (2, 300), device="cuda")
with torch.no_grad():
    M.use_sdpa(False); ref = m(x).float()
    M.use_sdpa(True);  fast = m(x).float()
d = (ref - fast).abs().max().item()
print(f"original vs SDPA attention, max |logit diff| = {d:.2e}"); assert d < 1e-2
del m, sd; torch.cuda.empty_cache()

In [ ]:
# 7 — generate on the official test set (both GPUs) -> /content/work/submission.zip
t0 = time.time()
!timeout -k 30 7200 torchrun --standalone --nproc_per_node={NGPU} generate.py gen_config.json {TEST_PATH} /content/work/run/test
assert os.path.exists("/content/work/run/test_submission.zip"), "GENERATION FAILED — see output above"
shutil.copy("/content/work/run/test_submission.zip", "/content/work/submission.zip")
shutil.copy("/content/work/run/test_prediction.json", "/content/work/prediction.json")
print(f"done in {(time.time()-t0)/60:.1f} min")
shutil.copy("/content/work/submission.zip", f"{DRIVE_DIR}/submission.zip"); print("copied to Drive:", f"{DRIVE_DIR}/submission.zip")

In [ ]:
# 8 — validate the zip
import zipfile
test = json.load(open(TEST_PATH, encoding="utf-8"))
with zipfile.ZipFile("/content/work/submission.zip") as z:
    assert z.namelist() == ["prediction.json"], z.namelist()
    pred = json.loads(z.read("prediction.json").decode("utf-8"))
assert set(pred) == set(test), "ID mismatch"
assert all(isinstance(v, dict) and isinstance(v.get("text"), str) for v in pred.values())
wc = [len(v["text"].split()) for v in pred.values()]
print(f"OK: {len(pred)} ids | words min/mean/max {min(wc)}/{sum(wc)/len(wc):.1f}/{max(wc)} | empty {sum(w == 0 for w in wc)}")
for k in list(pred)[:2]: print(f"  {k}: {pred[k]['text'][:150]}...")
from google.colab import files; files.download("/content/work/submission.zip")